# podcastAINews — Google Colab Runner
Generate an AI-powered podcast episode from the latest news on any topic.

In [ ]:
# Cell 1 — Clone or pull repo
import os, sys

REPO = "/content/podcastAINews"

if os.path.exists(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone https://github.com/TolsmA01/podcastAINews.git {REPO}

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('Working dir:', os.getcwd())

In [ ]:
# Cell 2 — Install dependencies
!apt-get install -y ffmpeg -q
!pip install -r /content/podcastAINews/requirements.txt -q

In [ ]:
# Cell 3 — Set OpenAI API key
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('API key set.')

In [ ]:
# Cell 4 — Set topic
TOPIC = 'artificial intelligence'  # <-- change this
print(f"Topic: '{TOPIC}'")

In [ ]:
# Cell 5 — Fetch news
# NOTE: os.chdir + sys.path are set explicitly here to avoid Colab path issues
import os, sys, importlib

REPO = '/content/podcastAINews'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import src.news_fetcher
importlib.reload(src.news_fetcher)
from src.news_fetcher import fetch_news

news_items = fetch_news(TOPIC)
print(f'\nFetched {len(news_items)} articles from {len({i.source for i in news_items})} sources:\n')
for item in news_items:
    print(f'  [{item.source}] {item.title}')

In [ ]:
# Cell 6 — Generate podcast script
import src.script_generator
importlib.reload(src.script_generator)
from src.script_generator import generate_script, _word_count, _estimate_minutes

script = generate_script(news_items, TOPIC)

words = _word_count(script)
mins  = _estimate_minutes(script)
print(f"\n{'='*60}")
print(f'PODCAST SCRIPT  |  {words} words  |  ~{mins:.1f} min estimated')
print('='*60)
print(script)

In [ ]:
# Cell 7 — Save transcript + sources
from datetime import datetime
from pathlib import Path

timestamp  = datetime.now().strftime('%Y-%m-%d_%H-%M')
output_dir = Path(REPO) / 'output'
output_dir.mkdir(exist_ok=True)

def _format_sources(items):
    lines = ['', 'Sources', '-------']
    for item in items:
        lines.append(f'- [{item.source}] {item.title}')
        if item.link:
            lines.append(f'  {item.link}')
    return '\n'.join(lines)

sources_block = _format_sources(news_items)
script_path   = output_dir / f'podcast_{timestamp}.txt'
script_path.write_text(script + sources_block, encoding='utf-8')

print(f'Transcript saved: {script_path}\n')
print(sources_block)

In [ ]:
# Cell 8 — Generate audio and play
import src.audio_generator
importlib.reload(src.audio_generator)
from src.audio_generator import generate_audio
from IPython.display import Audio, display

audio_path = output_dir / f'podcast_{timestamp}.mp3'
generate_audio(script, audio_path)

print(f'\nAudio saved: {audio_path}')
display(Audio(str(audio_path)))